# Final figure: standard_extended residuals + event arrows
**Purpose.** Re-fit the linear model on extended data (2000-2021 train / 2022-01-01 → 2026-01-01 test) and overlay event arrows on the residual panel for the 17 macro events identified by the |z|-screen + 2 manual GFC adds. 

**Inputs.** Yahoo Finance `^GSPC`, `^VIX` 1995-01-01 → 2026-01-01.

**Expected runtime.** ~3 min 

**Dependencies.** `code_section6.motivation`, `code_section6.sandbox.sanity_check.compute`, `code_guyon.empirical_study.main_function`.

## Imports + fit

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import code_section6.motivation as M
from code_section6.sandbox.sanity_check.compute import compute_residuals, rolling_zscore
from empirical_study.main_function import perform_empirical_study

In [ ]:
TRAIN_START = pd.Timestamp('2000-01-01')
TEST_START  = pd.Timestamp('2022-01-01')
TEST_END    = pd.Timestamp('2026-01-01')

In [ ]:
spx, vix = M.fetch_spx_vix_yahoo('1995-01-01', '2026-01-01')
sol = perform_empirical_study(
    vol=vix, index=spx, p=1, tspl=True,
    setting=[(1, 1), (2, 1/2)],
    train_start_date=TRAIN_START, test_start_date=TEST_START, test_end_date=TEST_END,
    max_delta=1000,
)
fitted_full = pd.concat([sol['train_pred'], sol['test_pred']]).sort_index()
residuals = compute_residuals(vix, fitted_full)
zscores   = rolling_zscore(residuals, window=252, min_periods=60)
rolling_sd = residuals.rolling(252, min_periods=60).std()
is_test = residuals.index >= TEST_START
print(f'fit done. train_R²={sol["train_r2"]:.4f}  test_R²={sol["test_r2"]:.4f}')
print(f'betas: {sol["opt_params"]}')

## Event annotations

In [ ]:
EVENTS = [
    ('2007-08-16', 'Quant Quake / BNP',           -55,  -55),
    ('2007-08-17', 'Fed emergency cut',           +60, -115),
    ('2008-10-13', 'GFC bounce (over-shot)',      -55,  +60),
    ('2008-10-24', 'GFC: VIX surge week',         +55,  -55),
    ('2010-05-07', 'Greek bailout / Flash Crash', -55,  -55),
    ('2010-05-20', 'Eurozone peak',               +60, -115),
    ('2011-08-09', 'US downgrade aftermath',        0,  +75),
    ('2014-10-16', 'Bullard / Ebola / Tsy flash',   0,  -80),
    ('2015-08-24', 'China devaluation crash',       0,  -55),
    ('2018-02-05', 'Volmageddon (XIV blow-up)',   -30,  -55),
    ('2020-02-28', 'COVID first spike',          -120, -125),
    ('2020-03-20', 'Fed credit facilities',       -80,  +80),
    ('2020-03-23', 'Fed unlimited QE',            +20, +160),
    ('2020-04-03', 'Post-QE collapse',           +120,  +80),
    ('2021-01-27', 'GameStop short squeeze',      +60, -115),
    ('2024-08-05', 'BoJ hike / yen carry unwind', -55,  -55),
    ('2025-04-08', 'Liberation Day tariffs',      +60, -115),
]
events_df = pd.DataFrame(EVENTS, columns=['date', 'label', 'ax', 'ay'])
events_df['date'] = pd.to_datetime(events_df['date'])
events_df['residual'] = events_df['date'].map(lambda d: residuals.asof(d))
events_df['resid_pp'] = (100 * events_df['residual']).round(2)
events_df['z'] = events_df['date'].map(lambda d: zscores.asof(d))
print(events_df[['date', 'label', 'resid_pp', 'z', 'ax', 'ay']].to_string(index=False))

## Build figure with arrows

In [ ]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    row_heights=[0.45, 0.55],
    subplot_titles=(
        'VIX (true) vs same-day fit (linear sigma = β₀ + β₁R1 + β₂sqrt.R2, train 2000-2021 / test 2022-2026)',
        'Residual = VIX_actual - VIX_fitted with rolling ±2sigma / ±2.5sigma bands and macro-event arrows'),
)
vix_aligned  = vix.reindex(residuals.index)
fitted_train = fitted_full.reindex(residuals.index[~is_test])
fitted_test  = fitted_full.reindex(residuals.index[ is_test])

fig.add_trace(go.Scatter(x=vix_aligned.index, y=100*vix_aligned, name='VIX (true)',
    line=dict(color='black', width=1.0)), row=1, col=1)
fig.add_trace(go.Scatter(x=fitted_train.index, y=100*fitted_train, name='Same-day fit (train)',
    line=dict(color='#1f77b4', width=1.2)), row=1, col=1)
fig.add_trace(go.Scatter(x=fitted_test.index, y=100*fitted_test, name='Same-day pred (test)',
    line=dict(color='#d62728', width=1.2)), row=1, col=1)

fig.add_trace(go.Scatter(x=residuals.index, y=100*residuals, name='Residual',
    line=dict(color='#666666', width=0.8)), row=2, col=1)
fig.add_trace(go.Scatter(x=rolling_sd.index, y= 200*rolling_sd, name='+2sigma rolling',
    line=dict(color='#aa3333', dash='dash', width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=rolling_sd.index, y=-200*rolling_sd, name='-2sigma rolling',
    line=dict(color='#aa3333', dash='dash', width=1), showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=rolling_sd.index, y= 250*rolling_sd, name='+2.5sigma rolling',
    line=dict(color='#aa0000', dash='dot', width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=rolling_sd.index, y=-250*rolling_sd, name='-2.5sigma rolling',
    line=dict(color='#aa0000', dash='dot', width=1), showlegend=False), row=2, col=1)

sep_ms = TEST_START.value // 10**6
for r in (1, 2):
    fig.add_shape(type='line', x0=sep_ms, x1=sep_ms, y0=0, y1=1,
                  yref=f'y{r} domain' if r == 2 else 'y domain',
                  xref=f'x{r}' if r == 2 else 'x',
                  line=dict(color='black', dash='dot', width=1))
fig.add_annotation(x=sep_ms, y=1.02, xref='x', yref='y domain',
                   text='train | test', showarrow=False, xanchor='left', font=dict(size=11))
fig.add_hline(y=0, line=dict(color='black', width=0.5), row=2, col=1)

for _, row in events_df.iterrows():
    date_ms = row['date'].value // 10**6
    resid_pp = row['resid_pp']
    color = '#d97706' if resid_pp > 0 else '#1d4ed8'
    fig.add_annotation(
        x=date_ms, y=resid_pp,
        text=row['label'],
        xref='x2', yref='y2',
        showarrow=True, arrowhead=2, arrowsize=1.0, arrowwidth=1.0, arrowcolor=color,
        ax=int(row['ax']), ay=int(row['ay']),
        font=dict(size=9, color=color),
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor=color, borderwidth=0.5, borderpad=2,
    )

fig.update_yaxes(title_text='VIX (%)', row=1, col=1)
fig.update_yaxes(title_text='Residual (pp)', row=2, col=1)
fig.update_layout(
    height=900, hovermode='x unified',
    title='§6.2 sanity check — same-day Guyon TSPL fit residuals with macro-event annotations (orange = under-predict spike; blue = over-predict / Fed deflation)',
    legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='left', x=0),
    margin=dict(l=60, r=20, t=130, b=40),
)
out_html = DESKTOP / 'sanity_check_annotated.html'
fig.write_html(out_html, include_plotlyjs='cdn')
print(f'saved: {out_html}')

## Static PDF mirror for LaTeX inclusion

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ORANGE, BLUE = '#d97706', '#1d4ed8'
CHAR_DAYS = 46.0   # label half-width scale (in days) for greedy row packing
ROW_STEP  = 3.6    # pp between stacked label rows in a band
BASE      = 16.5   # pp of the innermost label row (just outside the data)


def _pack_rows(sub):
    """Greedy row assignment so labels in one band never horizontally overlap.

    Walk events left-to-right; drop each into the lowest row whose last label
    ends before this one starts (estimated width = CHAR_DAYS * len(label)).
    """
    rows_right, out = [], {}
    for i in sub.sort_values('date').index:
        x = mdates.date2num(sub.loc[i, 'date'])
        half = 0.5 * CHAR_DAYS * len(sub.loc[i, 'label'])
        r = 0
        while r < len(rows_right) and rows_right[r] > x - half - 30:
            r += 1
        if r == len(rows_right):
            rows_right.append(x + half)
        else:
            rows_right[r] = x + half
        out[i] = r
    return out


fig_mpl, axs = plt.subplots(2, 1, figsize=(13, 8.2), sharex=True,
                            gridspec_kw={'height_ratios': [0.40, 0.60]})

# Top panel
axs[0].plot(vix_aligned.index,  100*vix_aligned,  color='black',   lw=0.9, label='VIX (true)')
axs[0].plot(fitted_train.index, 100*fitted_train, color='#1f77b4', lw=1.0, label='Same-day fit (train)')
axs[0].plot(fitted_test.index,  100*fitted_test,  color='#d62728', lw=1.0, label='Same-day pred (test)')
axs[0].axvline(TEST_START, color='black', linestyle=':', linewidth=0.8)
axs[0].text(TEST_START, axs[0].get_ylim()[1] * 0.97, ' train | test', fontsize=8, va='top')
axs[0].set_ylabel('VIX (%)')
axs[0].set_title('Same-day Guyon TSPL fit — train 2000-2021 / test 2022-2026 (linear $\\sigma = \\beta_0 + \\beta_1 R_1 + \\beta_2 \\sqrt{R_2}$)', fontsize=10)
axs[0].legend(loc='upper left', fontsize=7, framealpha=0.9)
axs[0].grid(True, alpha=0.3)

# Bottom panel
axs[1].plot(residuals.index,    100*residuals,    color='#666666', lw=0.6, label='Residual')
axs[1].plot(rolling_sd.index,  200*rolling_sd,    color='#aa3333', linestyle='--', lw=0.7, label=r'$\pm 2\sigma$ rolling')
axs[1].plot(rolling_sd.index, -200*rolling_sd,    color='#aa3333', linestyle='--', lw=0.7)
axs[1].plot(rolling_sd.index,  250*rolling_sd,    color='#aa0000', linestyle=':', lw=0.7, label=r'$\pm 2.5\sigma$ rolling')
axs[1].plot(rolling_sd.index, -250*rolling_sd,    color='#aa0000', linestyle=':', lw=0.7)
axs[1].axvline(TEST_START, color='black', linestyle=':', linewidth=0.8)
axs[1].axhline(0, color='black', linewidth=0.4)
axs[1].set_ylabel('Residual (pp)')
axs[1].set_xlabel('Date')
axs[1].set_ylim(-26, 29)
axs[1].legend(loc='upper left', fontsize=7, framealpha=0.9)
axs[1].grid(True, alpha=0.3)

# Event arrows: orange (under-predict, +resid) pinned to a top band, blue
# (over-predict, -resid) to a bottom band. Each band is greedy-packed into
# staggered rows so labels never overlap; a near-vertical arrow links each
# label to its residual spike.
events_df['is_up'] = events_df['resid_pp'] > 0
for is_up, color, sign in [(True, ORANGE, +1), (False, BLUE, -1)]:
    sub = events_df[events_df['is_up'] == is_up]
    rows = _pack_rows(sub)
    for i in sub.index:
        axs[1].annotate(
            sub.loc[i, 'label'],
            xy=(sub.loc[i, 'date'], sub.loc[i, 'resid_pp']),
            xytext=(sub.loc[i, 'date'], sign * (BASE + ROW_STEP * rows[i])),
            textcoords='data', ha='center', va='center',
            fontsize=6.5, color=color,
            arrowprops=dict(arrowstyle='->', color=color, lw=0.5, shrinkA=3, shrinkB=3),
            bbox=dict(boxstyle='round,pad=0.22', fc='white', ec=color, lw=0.45, alpha=0.92),
        )

axs[1].set_xlim(pd.Timestamp('1999-06-01'), pd.Timestamp('2026-10-01'))
plt.tight_layout()
out_pdf = pathlib.Path('/Users/idemskii/Desktop/projects/thesis/thesis_main/images_section_6/sanity_check_annotated.pdf')
out_pdf.parent.mkdir(parents=True, exist_ok=True)
fig_mpl.savefig(out_pdf, bbox_inches='tight')
print(f'saved: {out_pdf}')
plt.close(fig_mpl)